<a href="https://colab.research.google.com/github/satin-vasita/DSPY-Practicals-Colab-Notebooks/blob/main/Miniproject1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Imports
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# CELL 2 — Load Data
df = pd.read_csv('customer_booking.csv')
print(df.shape)
df.head()


(50000, 14)


,num_passengers,sales_channel,trip_type,purchase_lead,length_of_stay,flight_hour,flight_day,route,booking_origin,wants_extra_baggage,wants_preferred_seat,wants_in_flight_meals,flight_duration,booking_complete
0,2,Internet,RoundTrip,262,19,7,Sat,AKLDEL,New Zealand,1,0,0,5.52,0
1,1,Internet,RoundTrip,112,20,3,Sat,AKLDEL,New Zealand,0,0,0,5.52,0
2,2,Internet,RoundTrip,243,22,17,Wed,AKLDEL,India,1,1,0,5.52,0
3,1,Internet,RoundTrip,96,31,4,Sat,AKLDEL,New Zealand,0,0,1,5.52,0
4,2,Internet,RoundTrip,68,22,15,Wed,AKLDEL,India,1,0,1,5.52,0


In [ ]:
# CELL 3 — Attribute Descriptions (Markdown Cell in Jupyter)
"""
| Column                | Dtype   | Description                               |
|-----------------------|---------|-------------------------------------------|
| num_passengers        | int64   | Number of passengers in the booking       |
| sales_channel         | object  | Booking channel: Internet / Mobile        |
| trip_type             | object  | RoundTrip / OneWay / CircleTrip           |
| purchase_lead         | int64   | Days between booking and flight           |
| length_of_stay        | int64   | Nights at destination                     |
| flight_hour           | int64   | Departure hour (0–23)                     |
| flight_day            | object  | Day of week of departure                  |
| route                 | object  | Origin-Destination route code             |
| booking_origin        | object  | Country from which booking was made       |
| wants_extra_baggage   | int64   | Add-on: extra baggage (0/1)               |
| wants_preferred_seat  | int64   | Add-on: preferred seat (0/1)              |
| wants_in_flight_meals | int64   | Add-on: in-flight meals (0/1)             |
| flight_duration       | float64 | Flight duration in hours                  |
| booking_complete      | int64   | TARGET: booking completed (0/1)           |
"""

'\n| Column                | Dtype   | Description                               |\n|-----------------------|---------|-------------------------------------------|\n| num_passengers        | int64   | Number of passengers in the booking       |\n| sales_channel         | object  | Booking channel: Internet / Mobile        |\n| trip_type             | object  | RoundTrip / OneWay / CircleTrip           |\n| purchase_lead         | int64   | Days between booking and flight           |\n| length_of_stay        | int64   | Nights at destination                     |\n| flight_hour           | int64   | Departure hour (0–23)                     |\n| flight_day            | object  | Day of week of departure                  |\n| route                 | object  | Origin-Destination route code             |\n| booking_origin        | object  | Country from which booking was made       |\n| wants_extra_baggage   | int64   | Add-on: extra baggage (0/1)               |\n| wants_preferred_seat  |

In [ ]:
# CELL 4 — Info, Missing Values, Duplicates
df.info()
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())
df = df.drop_duplicates()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   num_passengers         50000 non-null  int64  
 1   sales_channel          50000 non-null  object 
 2   trip_type              50000 non-null  object 
 3   purchase_lead          50000 non-null  int64  
 4   length_of_stay         50000 non-null  int64  
 5   flight_hour            50000 non-null  int64  
 6   flight_day             50000 non-null  object 
 7   route                  50000 non-null  object 
 8   booking_origin         50000 non-null  object 
 9   wants_extra_baggage    50000 non-null  int64  
 10  wants_preferred_seat   50000 non-null  int64  
 11  wants_in_flight_meals  50000 non-null  int64  
 12  flight_duration        50000 non-null  float64
 13  booking_complete       50000 non-null  int64  
dtypes: float64(1), int64(8), object(5)
memory usage: 5.3+ 

In [ ]:
# CELL 5 — Statistical Summary
df.describe()

,num_passengers,purchase_lead,length_of_stay,flight_hour,wants_extra_baggage,wants_preferred_seat,wants_in_flight_meals,flight_duration,booking_complete
count,49281.000000,49281.000000,49281.000000,49281.000000,49281.000000,49281.000000,49281.000000,49281.000000,49281.000000
mean,1.590187,84.723281,23.053976,9.070676,0.668229,0.295631,0.426635,7.279974,0.149977
std,1.016538,90.410229,33.832149,5.413099,0.470854,0.456331,0.494593,1.496390,0.357052
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.670000,0.000000
25%,1.000000,21.000000,5.000000,5.000000,0.000000,0.000000,0.000000,5.620000,0.000000
50%,1.000000,51.000000,17.000000,9.000000,1.000000,0.000000,0.000000,7.570000,0.000000
75%,2.000000,115.000000,28.000000,13.000000,1.000000,1.000000,1.000000,8.830000,0.000000
max,9.000000,867.000000,778.000000,23.000000,1.000000,1.000000,1.000000,9.500000,1.000000


In [ ]:
# CELL 6 — Outlier Capping (IQR Winsorization)
# Capping instead of dropping to preserve dataset size.
# Applied to continuous features with known extreme values.
def cap_outliers(df, col):
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = df[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)
    return df

for col in ['purchase_lead', 'length_of_stay', 'flight_duration']:
    df = cap_outliers(df, col)
print("Outliers capped for: purchase_lead, length_of_stay, flight_duration")


Outliers capped for: purchase_lead, length_of_stay, flight_duration


In [ ]:
# CELL 7 — Feature Selection
# DROPPED: route (too many unique values, high cardinality adds noise)
#          booking_origin (same reason, 100+ countries without geo-grouping)
df = df.drop(columns=['route', 'booking_origin'])
print("Remaining features:", df.columns.tolist())

Remaining features: ['num_passengers', 'sales_channel', 'trip_type', 'purchase_lead', 'length_of_stay', 'flight_hour', 'flight_day', 'wants_extra_baggage', 'wants_preferred_seat', 'wants_in_flight_meals', 'flight_duration', 'booking_complete']


In [ ]:
# CELL 8 — Encoding Categorical Variables
# sales_channel → binary label encoding
df['sales_channel'] = df['sales_channel'].map({'Internet': 0, 'Mobile': 1})
# flight_day → ordinal (Mon=0 to Sun=6)
df['flight_day'] = df['flight_day'].map({'Mon':0,'Tue':1,'Wed':2,'Thu':3,'Fri':4,'Sat':5,'Sun':6})

# trip_type → one-hot encoding
df = pd.get_dummies(df, columns=['trip_type'], drop_first=False)

print("Encoding complete.")
df.head()

Encoding complete.


,num_passengers,sales_channel,purchase_lead,length_of_stay,flight_hour,flight_day,wants_extra_baggage,wants_preferred_seat,wants_in_flight_meals,flight_duration,booking_complete,trip_type_CircleTrip,trip_type_OneWay,trip_type_RoundTrip
0,2,0,256,19.0,7,5,1,0,0,5.52,0,False,False,True
1,1,0,112,20.0,3,5,0,0,0,5.52,0,False,False,True
2,2,0,243,22.0,17,2,1,1,0,5.52,0,False,False,True
3,1,0,96,31.0,4,5,0,0,1,5.52,0,False,False,True
4,2,0,68,22.0,15,2,1,0,1,5.52,0,False,False,True


In [ ]:
# CELL 9 — Save Cleaned Data
df.to_csv('cleaned_airline_data.csv', index=False)
print("Saved as cleaned_airline_data.csv | Shape:", df.shape)
print(df['booking_complete'].value_counts())

Saved as cleaned_airline_data.csv | Shape: (49281, 14)
booking_complete
0    41890
1     7391
Name: count, dtype: int64
